# 02b — Strategy B: Pix2Pix from scratch (paper-faithful)

Identical architecture, loss, and data pipeline as `02a_pix2pix_finetune.ipynb`, but **no pretrained warm-start** — random init. This lets us quantify how much the huggan warm-start actually buys us.

Hyper-differences vs 02a: `LR = 2e-4` (paper default), `EPOCHS = 200` with early-stop on `val_L1` plateau.

This notebook re-imports the model classes from 02a by exec-ing its definitions — avoids duplicating 200 lines of model code. If you refactor 02a's model, those changes apply here automatically.

In [ ]:
from __future__ import annotations
import sys, json
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

NB_DIR = Path.cwd()
sys.path.insert(0, str(NB_DIR))
import dstroke_utils as d

# Exec the code cells of 02a to pull in UNetGenerator, PatchDiscriminator, SynthPairs,
# trapped_ball_batch, paths (SYNTH/OUT), etc. The warm-start cell will no-op here
# because we re-instantiate G below.
NB_A = NB_DIR / '02a_pix2pix_finetune.ipynb'
nb = json.load(open(NB_A))
code_to_exec = '\n\n'.join(''.join(c['source']) for c in nb['cells'] if c['cell_type'] == 'code'
                           and 'history = {' not in ''.join(c['source'])
                           and 'training curves' not in ''.join(c['source']))
exec(code_to_exec, globals())

CKPT = OUT / 'strategy_B_pix2pix_scratch.pt'
print('Device:', DEVICE, '| train:', len(train_ds), ' val:', len(val_ds), ' ckpt:', CKPT)

In [ ]:
# Re-instantiate G and D with random init — this overrides the warm-started versions from 02a.
G = UNetGenerator(in_ch=3, out_ch=1).to(DEVICE)
D = PatchDiscriminator(in_ch=5).to(DEVICE)

# Apply Pix2Pix-style init (normal 0, 0.02) — paper default.
def init_weights(m):
    cn = m.__class__.__name__
    if 'Conv' in cn:
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif 'BatchNorm' in cn:
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.constant_(m.bias, 0)

G.apply(init_weights); D.apply(init_weights)

LR = 2e-4
EPOCHS = 200
PATIENCE = 15  # early-stop on val_L1
LAMBDA_L1 = 100.0

opt_G = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss(); l1 = nn.L1Loss()

history_B = {'d_loss': [], 'g_loss': [], 'g_l1': [], 'val_l1': []}
best_val = float('inf'); bad = 0

for epoch in range(1, EPOCHS+1):
    G.train(); D.train()
    d_ep, g_ep, l1_ep = 0.0, 0.0, 0.0
    for x, y in tqdm(train_dl, desc=f'epoch {epoch}/{EPOCHS}'):
        x = x.to(DEVICE); y = y.to(DEVICE)
        with torch.no_grad():
            g_out = G(x); t_fake = trapped_ball_batch(g_out); t_real = trapped_ball_batch(y)
        d_real = run_D(x, y, t_real); d_fake = run_D(x, g_out, t_fake)
        loss_D = 0.5*(bce(d_real, torch.ones_like(d_real)) + bce(d_fake, torch.zeros_like(d_fake)))
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()
        g_out = G(x)
        with torch.no_grad():
            t_fake = trapped_ball_batch(g_out)
        d_fake2 = run_D(x, g_out, t_fake)
        loss_G = bce(d_fake2, torch.ones_like(d_fake2)) + l1(g_out, y)*LAMBDA_L1
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()
        d_ep += float(loss_D); g_ep += float(loss_G); l1_ep += float(l1(g_out, y)*LAMBDA_L1)
    n = len(train_dl)
    G.eval(); vl = 0.0; vn = 0
    with torch.no_grad():
        for x, y in val_dl:
            x=x.to(DEVICE); y=y.to(DEVICE); vl += float(l1(G(x), y)); vn += 1
    cur_val = vl / max(vn, 1)
    history_B['d_loss'].append(d_ep/n); history_B['g_loss'].append(g_ep/n)
    history_B['g_l1'].append(l1_ep/n); history_B['val_l1'].append(cur_val)
    print(f'epoch {epoch}: val_L1={cur_val:.3f}  (best {best_val:.3f})')
    if cur_val < best_val - 1e-4:
        best_val = cur_val; bad = 0
        torch.save({'G': G.state_dict(), 'D': D.state_dict(), 'history': history_B, 'epoch': epoch}, CKPT)
    else:
        bad += 1
        if bad >= PATIENCE:
            print(f'early-stop at epoch {epoch}, best val_L1 = {best_val:.3f}')
            break
    if epoch % 10 == 0:
        sample_preview(epoch)
print('saved best to', CKPT)

In [ ]:
# side-by-side learning curves: 02a (fine-tune) vs 02b (from scratch)
try:
    A = torch.load(OUT / 'strategy_A_pix2pix_finetune.pt', map_location='cpu')['history']
except FileNotFoundError:
    A = None
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(history_B['val_l1'], label='B (scratch)')
if A: ax[0].plot(A['val_l1'], label='A (fine-tune)')
ax[0].set_title('val L1 over epochs'); ax[0].legend()
ax[1].plot(history_B['d_loss'], label='B D'); ax[1].plot(history_B['g_loss'], label='B G_gan')
if A: ax[1].plot(A['d_loss'], label='A D', linestyle='--'); ax[1].plot(A['g_loss'], label='A G_gan', linestyle='--')
ax[1].set_title('adversarial losses'); ax[1].legend(); plt.show()